# 04 - Cohort Retention Analysis

This notebook analyzes customer cohort retention:
- Retention matrix
- Retention curves over time
- References `outputs/metrics/cohort_summary.json`
- Displays generated cohort heatmap and retention curves PNGs

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("Libraries loaded successfully.")

## 2. Load Cohort Retention Data

In [ ]:
processed_path = Path('../data/processed')

cohort_path = processed_path / 'cohort_retention.csv'

if cohort_path.exists():
    cohort_retention = pd.read_csv(cohort_path, index_col=0)
    print(f'Cohort retention matrix loaded: {cohort_retention.shape}')
    print(f'\nFirst few rows:')
    print(cohort_retention.head())
else:
    print("cohort_retention.csv not found. Computing from orders data...")
    orders = pd.read_csv(processed_path / 'orders_processed.csv')
    orders["order_date"] = pd.to_datetime(orders["order_date"])
    
    orders['order_month'] = orders['order_date'].dt.to_period('M')
    customer_cohort = orders.groupby('customer_id')['order_date'].min().dt.to_period('M')
    orders = orders.merge(customer_cohort.rename('cohort'), on='customer_id')
    orders['cohort_period'] = (orders['order_month'] - orders['cohort']).apply(lambda x: x.n)
    
    cohort_data = orders.groupby(['cohort', 'cohort_period'])['customer_id'].nunique().reset_index()
    cohort_data.columns = ['cohort', 'period', 'customers']
    
    cohort_sizes = cohort_data.groupby('cohort')['customers'].first()
    cohort_data['retention'] = cohort_data.apply(
        lambda row: row['customers'] / cohort_sizes[row['cohort']], axis=1
    )
    
    cohort_retention = cohort_data.pivot(index='cohort', columns='period', values='retention')
    print(f'Cohort retention matrix computed: {cohort_retention.shape}')

## 3. Retention Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(
    cohort_retention,
    annot=True,
    fmt='.2%',
    cmap='YlGnBu',
    ax=ax,
    cbar_kws={'label': 'Retention Rate'}
)
ax.set_title('Customer Cohort Retention Matrix', fontsize=14)
ax.set_xlabel('Periods Since First Purchase', fontsize=12)
ax.set_ylabel('Cohort (First Purchase Month)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Retention Curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

n_cohorts_to_plot = min(8, len(cohort_retention))
selected_cohorts = cohort_retention.head(n_cohorts_to_plot)

for idx, (cohort, row) in enumerate(selected_cohorts.iterrows()):
    valid = row.dropna()
    ax.plot(valid.index, valid.values, marker='o', label=str(cohort), alpha=0.7)

ax.set_title('Retention Curves by Cohort', fontsize=14)
ax.set_xlabel('Periods Since First Purchase', fontsize=12)
ax.set_ylabel('Retention Rate', fontsize=12)
ax.legend(title='Cohort', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Average Retention by Period

In [ ]:
avg_retention = cohort_retention.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
avg_retention.plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Average Retention Rate by Period', fontsize=14)
ax.set_xlabel('Period', fontsize=12)
ax.set_ylabel('Average Retention Rate', fontsize=12)
ax.set_xticklabels([f'Period {int(x)}' for x in avg_retention.index], rotation=45)

for i, v in enumerate(avg_retention.values):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Cohort Summary Report

In [ ]:
metrics_path = Path('../outputs/metrics')

cohort_summary_path = metrics_path / 'cohort_summary.json'
if cohort_summary_path.exists():
    with open(cohort_summary_path, "r") as f:
        cohort_summary = json.load(f)
    print("COHORT SUMMARY")
    print("=" * 60)
    print(json.dumps(cohort_summary, indent=2, default=str))
else:
    print("cohort_summary.json not found.")

## 7. Display Generated Figures

In [ ]:
from IPython.display import Image, display

figures_path = Path('../outputs/figures')

cohort_heatmap = figures_path / 'cohort_heatmap.png'
if cohort_heatmap.exists():
    print('Cohort Heatmap:')
    display(Image(filename=str(cohort_heatmap)))
else:
    print("cohort_heatmap.png not found in outputs/figures/.")

In [ ]:
retention_curves = figures_path / 'retention_curves.png'
if retention_curves.exists():
    print('Retention Curves:')
    display(Image(filename=str(retention_curves)))
else:
    print("retention_curves.png not found in outputs/figures/.")

## 8. Summary

In [ ]:
print('Cohort Retention Summary')
print("=" * 60)
if not cohort_retention.empty:
    print(f"Number of cohorts: {len(cohort_retention)}")
    if 1 in cohort_retention.columns:
        print(f"Average period 1 retention: {cohort_retention[1].mean():.1%}")
    if 3 in cohort_retention.columns:
        print(f"Average period 3 retention: {cohort_retention[3].mean():.1%}")
    if 6 in cohort_retention.columns:
        print(f"Average period 6 retention: {cohort_retention[6].mean():.1%}")
    print(f"Best performing cohort: {cohort_retention.index[0]}")
else:
    print("No cohort data available.")